# 07 — Source-Confound Diagnostic (multi-turn)

**Question this notebook answers:** how much of the multi-turn *"harm"* signal is really just **source** detection?

In the multi-turn dataset, every harmful example comes from **SafeDial** and every benign example comes from **LMSYS**. So the `harm` label is perfectly confounded with the data *source*. A classifier could score high on "harm" purely by learning *"is this a SafeDial conversation or an LMSYS conversation"* — topic and style — without detecting anything jailbreak-specific.

This notebook trains a classifier to predict **source** (which is identical to `harm`) and reports how easily separable the two sources are. That number is the **ceiling of trust** for any multi-turn harm result.

## Method

- Reuse the **exact** TF-IDF + Logistic Regression pipeline from `06_flo_baseline_single_full.ipynb`, so the diagnostic is apples-to-apples with the baseline.
- Train on the existing `multiturn_*_train` split, evaluate on `multiturn_*_test`.
- Report **ROC-AUC** (threshold-independent) and inspect the top features.

**How to read the AUC:**
- ≈ **0.5** → sources genuinely hard to tell apart → the multi-turn task is honest.
- ≈ **0.95–1.0** → sources trivially separable → multi-turn "harm detection" is mostly source detection; fix the labels before trusting any comparison.

> **Prereq:** `multiturn_X_*.csv` / `multiturn_Y_*.csv` must be in `data/processed/`. They are gitignored — get them from David or regenerate.

In [ ]:
# TODO: imports
import numpy as mp
from pyprojroot import here
from sklearn.feature_extraction.text import TFidVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, roc_auc_score


# TODO: paths & constants

REPO_ROOT = here()
PROCESSED = REPO_ROOT / "data" / "processed"
# SEED = 1234

## 1. Load the multi-turn splits

`multiturn_X_{split}.csv` has columns `conversation_id`, `conversation`; `multiturn_Y_{split}.csv` has `harm` (bool). The `harm` label **is** the source (SafeDial = harmful = 1, LMSYS = benign = 0), so we use it directly as the source target.

In [ ]:
# TODO: write a helper load_multi(split) that reads
#   multiturn_X_{split}.csv  -> conversation text
#   multiturn_Y_{split}.csv  -> harm label
# and returns (conversation Series, harm-as-int Series)

# TODO: load train and test
# X_train, y_train = load_multi("train")
# X_test,  y_test  = load_multi("test")

# Sanity: sizes + source balance (expect ~0.5)
# print(len(X_train), len(X_test), round(y_train.mean(), 3))

## 2. Vectorize (match the baseline exactly)

Use the **same** knobs as `06`: `ngram_range=(1,2)`, `min_df=5`, `max_df=0.9`, `sublinear_tf=True`, `stop_words="english"`. Fit on train only; transform test — no leakage.

In [ ]:
# TODO: build the TfidfVectorizer with the baseline's config
# vectorizer = ...

# TODO: fit_transform on X_train, transform on X_test
# Xtr = ...
# Xte = ...
# print("vocab size:", len(vectorizer.vocabulary_))

## 3. Fit the source classifier

Logistic Regression, same settings as the baseline (raise `max_iter` if it warns about convergence).

In [ ]:
# TODO: clf = LogisticRegression(...)
# TODO: clf.fit(Xtr, y_train)

## 4. Evaluate — the ceiling-of-trust number

AUC uses **probabilities** (`predict_proba(...)[:, 1]`), not the 0/1 predictions. This AUC is the headline: how well source is separable = the upper bound on how much of any multi-turn "harm" score could just be source detection.

In [ ]:
# TODO: test_prob = clf.predict_proba(Xte)[:, 1]
# TODO: test_pred = clf.predict(Xte)

# TODO: print(classification_report(y_test, test_pred, target_names=["LMSYS", "SafeDial"]))
# TODO: print("Source AUC:", roc_auc_score(y_test, test_prob))

## 5. The smoking gun — what is it keying on?

List the top tokens pushing toward each source. **If the strongest source-predictors are benign topical/style words** (small talk, formatting artifacts, names) **rather than harm words, that is direct evidence the split is learnable from non-harm cues** — the confound made visible.

Hint: `weights = clf.coef_[0]`; `np.argsort(weights)` → most negative = LMSYS end, most positive = SafeDial end.

In [ ]:
# TODO: feature_names = vectorizer.get_feature_names_out()
# TODO: weights = clf.coef_[0]

# TODO: show top ~20 tokens toward SafeDial (harmful) and top ~20 toward LMSYS (benign)
# Read the lists: harm words, or topic/style?

## 6. Interpretation & next step

Record two things for the team:
1. **Source AUC** — the ceiling of trust. Cite it wherever a multi-turn harm number appears.
2. **Top features** — whether separation rides on harm or on topic/style.

**Decision gate:**
- Low AUC (≈0.5) → multi-turn task is honest → the *train-single vs train-multi* comparison is worth building.
- High AUC (→1.0) → fix the multi-turn labels (harmful **and** benign from the *same* source) before trusting any multi-turn comparison.